# pgvector Connection Test

Run cells top to bottom. Each cell is one check.

If no pgvector container is running, start one first:
```bash
docker run -d \
  --name pgvector \
  -e POSTGRES_PASSWORD=postgres \
  -p 5432:5432 \
  pgvector/pgvector:pg16
```

## 1. Check Docker — is the container running?

In [1]:
import subprocess
result = subprocess.run(["docker", "ps", "--format", "table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}"],
                        capture_output=True, text=True)
print(result.stdout or "(no containers running)")

NAMES                        IMAGE                                                                             STATUS                PORTS
pgvector                     pgvector/pgvector:pg16                                                            Up 58 seconds         0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp
test-xml-tools-connector-1   773267254890.dkr.ecr.us-east-1.amazonaws.com/workato/xml-tools-connector:1.27.0   Up 3 days (healthy)   0.0.0.0:18088->8088/tcp, [::]:18088->8088/tcp
dev-xml-tools-connector-1    773267254890.dkr.ecr.us-east-1.amazonaws.com/workato/xml-tools-connector:1.27.0   Up 3 days (healthy)   0.0.0.0:8088->8088/tcp, [::]:8088->8088/tcp
test-zookeeper-1             confluentinc/cp-zookeeper:7.6.4                                                   Up 3 days (healthy)   2181/tcp, 2888/tcp, 3888/tcp
dev-zookeeper-1              confluentinc/cp-zookeeper:7.6.4                                                   Up 3 days (healthy)   0.0.0.0:2181->2181/tcp, [::]:2181

## 2. Connection settings

Edit these if your container uses different values.

In [2]:
PG_HOST     = "localhost"
PG_PORT     = 5432
PG_DATABASE = "postgres"
PG_USER     = "postgres"
PG_PASSWORD = "postgres"

print(f"Connecting to {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}")

Connecting to postgres@localhost:5432/postgres


## 3. Connect to Postgres

In [4]:
import psycopg2

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT,
    dbname=PG_DATABASE, user=PG_USER, password=PG_PASSWORD
)
cur = conn.cursor()

cur.execute("SELECT version();")
print("Connected:", cur.fetchone()[0])

Connected: PostgreSQL 16.13 (Debian 16.13-1.pgdg12+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


## 4. Check pgvector extension

In [5]:
cur.execute("SELECT extname, extversion FROM pg_extension WHERE extname = 'vector';")
row = cur.fetchone()
if row:
    print(f"pgvector extension installed — version {row[1]}")
else:
    print("pgvector NOT installed — running CREATE EXTENSION ...")
    cur.execute("CREATE EXTENSION vector;")
    conn.commit()
    print("Done.")

pgvector NOT installed — running CREATE EXTENSION ...
Done.


## 5. Check which tables exist

In [6]:
cur.execute("""
    SELECT table_name
    FROM   information_schema.tables
    WHERE  table_schema = 'public'
    ORDER  BY table_name;
""")
tables = [r[0] for r in cur.fetchall()]
if tables:
    print("Tables in public schema:")
    for t in tables:
        print(" ", t)
else:
    print("No tables yet — run setup_schema.sql first:")
    print("  psql -h localhost -p 5432 -U postgres -d postgres -f 06-pgvector/setup_schema.sql")

No tables yet — run setup_schema.sql first:
  psql -h localhost -p 5432 -U postgres -d postgres -f 06-pgvector/setup_schema.sql


## 6. Row counts (if tables exist)

In [ ]:
expected_tables = [
    "recipes",
    "embeddings_text_embedding_3_small",
    "embeddings_text_embedding_3_large",
]

for table in expected_tables:
    if table in tables:
        cur.execute(f"SELECT COUNT(*) FROM {table};")
        count = cur.fetchone()[0]
        print(f"  {table:<45}  {count:>5} rows")
    else:
        print(f"  {table:<45}  (not created yet)")

## 7. Quick vector smoke test

Verifies that the vector type works by creating and querying a tiny temp table.

In [ ]:
cur.execute("""
    CREATE TEMP TABLE _vec_test (id INT, v vector(3));
    INSERT INTO _vec_test VALUES (1, '[1,0,0]'), (2, '[0,1,0]'), (3, '[0,0,1]');
""")

query_vec = '[1,0,0]'
cur.execute(f"""
    SELECT id,
           1 - (v <=> '{query_vec}'::vector) AS cosine_similarity
    FROM   _vec_test
    ORDER  BY v <=> '{query_vec}'::vector
    LIMIT  3;
""")

print(f"Nearest neighbours to {query_vec}:")
for row in cur.fetchall():
    print(f"  id={row[0]}  cosine_similarity={row[1]:.4f}")

print("\nvector operations working correctly.")

## 8. Peek at recipes table (if loaded)

In [ ]:
import pandas as pd

if "recipes" in tables:
    cur.execute("""
        SELECT recipe_uid, author_id, connectors, step_count,
               LEFT(text, 80) AS text_preview
        FROM   recipes
        LIMIT  5;
    """)
    cols = [d[0] for d in cur.description]
    df = pd.DataFrame(cur.fetchall(), columns=cols)
    display(df)
else:
    print("recipes table not found — run ingest_recipes.py first.")

## 9. Close connection

In [7]:
cur.close()
conn.close()
print("Connection closed.")

Connection closed.
